# 143 SGD code -- mini-batch gradient descent

Lean PyTorch loop for **mini-batch GD** (what DL usually calls SGD).

Companion lectures: `144_SGD_concept.py`, `145_SGD_training.py`

| Term | Meaning |
|------|---------|
| `w` | model parameters (a **condition** for the loss) |
| `L(... ; w)` | full-batch loss on all data, given `w` |
| mini-batch loss | loss on `b` rows -- builds the gradient for one SGD step |
| SGD step | one `optimizer.step()` |


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

N, D, B, LR, EPOCHS = 512, 4, 32, 0.05, 20

X = torch.randn(N, D)
w_true = torch.tensor([1.4, -0.8, 0.55, 0.2])
y = X @ w_true + 0.3 * torch.randn(N)

model = nn.Linear(D, 1, bias=False)
opt = torch.optim.SGD(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

@torch.no_grad()
def full_batch_loss():
    """L(... ; w): mean loss on ALL N examples."""
    return loss_fn(model(X).squeeze(), y).item()


## Training loop

Each SGD step:
1. sample `b` rows
2. compute **mini-batch loss**
3. `backward` + `step`

`L(... ; w)` is evaluated only to monitor progress (not needed to take the SGD step).


In [ ]:
history = [full_batch_loss()]

for epoch in range(EPOCHS):
    perm = torch.randperm(N)
    for i in range(0, N, B):
        idx = perm[i : i + B]
        loss = loss_fn(model(X[idx]).squeeze(), y[idx])  # mini-batch loss
        opt.zero_grad()
        loss.backward()
        opt.step()                                       # SGD step
    history.append(full_batch_loss())
    print(f"epoch {epoch + 1:02d}  L(... ; w) = {history[-1]:.4f}")

print("learned w:", model.weight.detach().squeeze().tolist())
print("true    w:", w_true.tolist())
